In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("/Users/subramanip/Downloads/DSML_Project/project1_nlp/data/processed/cleaned_dataset.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head(2))

(193533, 11)
['Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Complaint Type', 'Descriptor', 'Incident Zip', 'Borough', 'text', 'department', 'clean_text']
              Created Date Closed Date Agency  \
0  2019-12-01T02:04:01.000         NaN    DOT   
1  2019-12-01T01:59:41.000         NaN   NYPD   

                       Agency Name      Complaint Type        Descriptor  \
0     Department of Transportation    Street Condition           Pothole   
1  New York City Police Department  Noise - Commercial  Loud Music/Party   

   Incident Zip    Borough                                 text  \
0       10001.0  MANHATTAN             Street Condition Pothole   
1       11223.0   BROOKLYN  Noise - Commercial Loud Music/Party   

                        department                        clean_text  
0     Department of Transportation          street condition pothole  
1  New York City Police Department  noise commercial loud musicparty  


In [2]:
# TF-IDF vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X = tfidf.fit_transform(df["clean_text"])
y = df["department"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Matrix shape:", X.shape)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Number of departments:", y.nunique())



Matrix shape: (193533, 2402)
Train size: 154826
Test size: 38707
Number of departments: 13


In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

print("✅ Model trained!")

✅ Model trained!


In [5]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9891

Classification Report:
                                                    precision    recall  f1-score   support

                              BCC - Brooklyn North       0.47      0.68      0.56       229
                              BCC - Brooklyn South       0.51      0.17      0.25       338
                               BCC - Staten Island       0.53      0.77      0.63       276
                           Department of Buildings       1.00      1.00      1.00      1721
                    Department of Consumer Affairs       1.00      1.00      1.00       204
            Department of Environmental Protection       1.00      1.00      1.00      2788
           Department of Health and Mental Hygiene       1.00      1.00      1.00       965
                   Department of Homeless Services       1.00      1.00      1.00       521
Department of Housing Preservation and Development       1.00      1.00      1.00     10099
                Department of Parks an

In [6]:
from sklearn.model_selection import cross_val_score
import numpy as np

cv_scores = cross_val_score(model, X, y, cv=5, scoring="f1_macro")

print("Cross-val Macro F1 scores:", cv_scores.round(4))
print("Mean:", round(cv_scores.mean(), 4))
print("Std:", round(cv_scores.std(), 4))

Cross-val Macro F1 scores: [0.8747 0.8813 0.8831 0.8828 0.8739]
Mean: 0.8791
Std: 0.004


In [7]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
joblib.dump(model, "models/department_classifier.pkl")

print("✅ TF-IDF vectorizer saved!")
print("✅ Department classifier saved!")

✅ TF-IDF vectorizer saved!
✅ Department classifier saved!


In [8]:
# Load saved models and test
tfidf_loaded = joblib.load("models/tfidf_vectorizer.pkl")
model_loaded = joblib.load("models/department_classifier.pkl")

# Test samples
test_samples = [
    "loud music party at night",
    "pothole on the street",
    "water pipe leaking",
    "garbage not collected",
    "broken traffic light"
]

for sample in test_samples:
    vec = tfidf_loaded.transform([sample])
    pred = model_loaded.predict(vec)[0]
    print(f"'{sample}' → {pred}")

'loud music party at night' → New York City Police Department
'pothole on the street' → Department of Transportation
'water pipe leaking' → Department of Environmental Protection
'garbage not collected' → Department of Parks and Recreation
'broken traffic light' → Department of Transportation
